In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window, DataFrame
import torch
import pandas as pd
import tiktoken

torch.manual_seed(123)

src_path = Path.cwd().parent / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to sys.path: {src_path}")

load_dotenv()  # reads .env file from the current directory
spark = SparkSession.builder.getOrCreate()

Added to sys.path: /home/jtv/code/jtviegas/languagemodels/notebook/src


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/16 14:03:50 WARN Utils: Your hostname, jtv, resolves to a loopback address: 127.0.1.1; using 192.168.0.160 instead (on interface wlp192s0)
26/06/16 14:03:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 14:03:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# constants

In [2]:
DEFAULT_EMBEDDINGS_DIMENSION = 256
DEFAULT_CONTEXT_LENGTH = 8
DEFAULT_STRIDE = 1

path_data = Path.cwd().parent / "data"
path_narratives = path_data / "faers" / "faers_narratives_small"
path_gpt2_settings = path_data / "model_weights" / "gpt2" / "124M" / "settings.pickle.gz"
path_gpt2_parameters = path_data / "model_weights" / "gpt2" / "124M" / "parameters.pickle.gz"

# data

In [3]:
def get_faers_narratives(n: int = 1000) -> pd.DataFrame:
    return spark.read.load(str(path_narratives)).limit(n).select("text").toPandas()

In [4]:
def get_dummy_corpus() -> pd.DataFrame:
    return pd.DataFrame({
        "text": [
            "The cat is on the table.",
            "The dog is in the garden.",
            "The bird is flying in the sky.",
            "The fish is swimming in the pond."
        ]
    })

# input preparation

In [5]:
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.eot_token)
print(tokenizer.n_vocab)

50256
50257


In [6]:
from tgedr_languagemodels.utils.utils_llm import harmonize_text_sequences

data = harmonize_text_sequences(get_dummy_corpus(), tokenizer=tokenizer, sequence_length=DEFAULT_CONTEXT_LENGTH)
context_length = len(data[0])
d = torch.tensor(data, dtype=torch.long)
print(data)
d

[[464, 3797, 318, 319, 262, 3084, 13, 50256], [464, 3290, 318, 287, 262, 11376, 13, 50256], [464, 6512, 318, 7348, 287, 262, 6766, 13], [464, 5916, 318, 14899, 287, 262, 16723, 13]]


tensor([[  464,  3797,   318,   319,   262,  3084,    13, 50256],
        [  464,  3290,   318,   287,   262, 11376,    13, 50256],
        [  464,  6512,   318,  7348,   287,   262,  6766,    13],
        [  464,  5916,   318, 14899,   287,   262, 16723,    13]])

In [7]:

from tgedr_languagemodels.gpt2.model import ModelConfig


config = ModelConfig( 
    vocabulary_size=tokenizer.n_vocab,
    embeddings_dimension=DEFAULT_EMBEDDINGS_DIMENSION,
    context_length=context_length,
    n_layers=12,
    drop_rate=0.1,
    stride=DEFAULT_STRIDE,
    n_heads=4
)

ImportError: cannot import name 'ModelConfig' from 'tgedr_languagemodels.gpt2.model' (/home/jtv/code/jtviegas/languagemodels/src/tgedr_languagemodels/gpt2/model.py)

In [ ]:
from tgedr_languagemodels.layers.blocks import MultiHeadAttention
from tgedr_languagemodels.gpt2.model import GPT2Model


model = GPT2Model(config)
print(model)

GPT2Model(
  (tok_emb): Embedding(50257, 256)
  (pos_emb): Embedding(8, 256)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=256, out_features=256, bias=False)
        (W_key): Linear(in_features=256, out_features=256, bias=False)
        (W_value): Linear(in_features=256, out_features=256, bias=False)
        (out_projection): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU()
          (2): Linear(in_features=1024, out_features=256, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_fea

In [ ]:
context_sentence = d[:, -context_length:]
print(context_sentence)
with torch.no_grad():
  logits = model(context_sentence)


tensor([[  464,  3797,   318,   319,   262,  3084,    13, 50256],
        [  464,  3290,   318,   287,   262, 11376,    13, 50256],
        [  464,  6512,   318,  7348,   287,   262,  6766,    13],
        [  464,  5916,   318, 14899,   287,   262, 16723,    13]])


In [ ]:
max_new_tokens = 3
for _ in range(max_new_tokens):
    idx_cond = idx[:, -context_size:]
    with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:  # 2
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)
        if temperature > 0.0:  # 3
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:  # 4
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next == eos_id:  # 5
            break
        idx = torch.cat((idx, idx_next), dim=1)
   

In [ ]:
from tgedr_languagemodels.utils.utils_llm import save_pickle_compressed, load_pickle_compressed

params = load_pickle_compressed(model_weights_parameters)
settings = load_pickle_compressed(model_weights_settings)

# model

In [ ]:
from tgedr_languagemodels.models import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"

NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024})
NEW_CONFIG.update({"qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

In [ ]:
from tgedr_languagemodels.utils.model_weights import load_weights_into_gpt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_weights_into_gpt(gpt, params)
gpt.to(device)

In [ ]:
from tgedr_languagemodels.utils.utils_llm import generate, text_to_token_ids, token_ids_to_text
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")


token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5,
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

# fine-tuning for classification

## getting the data

In [ ]:
path_spam = path_data / "sms_spam_collection"
zipfile_url = str(path_data / "sms_spam_collection.zip")
spam_tsv_url = str(path_spam / "SMSSpamCollection.tsv")
spam_url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

In [ ]:
import urllib.request
import zipfile
import os
from pathlib import Path

def download_and_unzip_spam_data(url, zip_path, path_spam, path_data):
    if Path(spam_tsv_url).exists():
        print(f"{spam_tsv_url} already exists. Skipping download "
              "and extraction."
        )
        return

    with urllib.request.urlopen(url) as response:    #1
        with open(zipfile_url, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zipfile_url, "r") as zip_ref:    #2
        zip_ref.extractall(path_spam)

    original_file_path = path_spam / "SMSSpamCollection"
    os.rename(original_file_path, spam_tsv_url)               #3
    print(f"File downloaded and saved as {spam_tsv_url}")

download_and_unzip_spam_data(spam_url, zipfile_url, path_spam, path_data)
#1 Downloads the file
#2 Unzips the file
#3 Adds a .tsv file extension

In [ ]:
import pandas as pd
df = pd.read_csv(
    spam_tsv_url, sep="\t", header=None, names=["Label", "Text"]
)
df      #1

In [ ]:
print(df["Label"].value_counts())

In [ ]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]     #1
    ham_subset = df[df["Label"] == "ham"].sample(
        num_spam, random_state=123
    )                                         #2
    balanced_df = pd.concat([
        ham_subset, df[df["Label"] == "spam"]
    ])                               #3
    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())
#1 Counts the instances of “spam”
#2 Randomly samples “ham” instances to match the number of “spam” instances
#3 Combines ham subset with “spam”

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

In [ ]:
def random_split(df, train_frac, validation_frac):

    df = df.sample(
        frac=1, random_state=123
    ).reset_index(drop=True)               #1
    train_end = int(len(df) * train_frac)          #2
    validation_end = train_end + int(len(df) * validation_frac)

 #3
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(
    balanced_df, 0.7, 0.1)                     #4
#1 Shuffles the entire DataFrame
#2 Calculates split indices
#3 Splits the DataFrame
#4 Test size is implied to be 0.2 as the remainder.

In [ ]:
train_df.to_csv(str(path_spam / "train.csv"), index=None)
validation_df.to_csv(str(path_spam / "validation.csv"), index=None)
test_df.to_csv(str(path_spam / "test.csv"), index=None)

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

In [ ]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, pd_df, tokenizer, max_length=None,
                 pad_token_id=50256):
        # self.data = pd.read_csv(csv_file)
        self.data = pd_df

 #1
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
 #2
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

 #3
        self.encoded_texts = [
            encoded_text + [pad_token_id] * 
            (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]


    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
#1 Pretokenizes texts
#2 Truncates sequences if they are longer than max_length
#3 Pads sequences to the longest sequence

In [ ]:
balanced_df

In [ ]:
train_dataset = SpamDataset(
    pd_df=train_df,
    max_length=None,
    tokenizer=tokenizer
)

In [ ]:
print(train_dataset.max_length)

In [ ]:
val_dataset = SpamDataset(
    pd_df=validation_df,
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    pd_df=test_df,
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0      #1 This setting ensures compatibility with most computers.
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [ ]:
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

In [ ]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

#### Initializing a model with pretrained weights

In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"
BASE_CONFIG = {
    "vocab_size": 50257,          #1
    "context_length": 1024,       #2
    "drop_rate": 0.0,             #3
    "qkv_bias": True              #4
}
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
#1 Vocabulary size
#2 Context length
#3 Dropout rate
#4 Query-key-value bias

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


def generate_text_simple(
    model,
    idx,  # 1
    max_new_tokens,
    context_size,
    temperature=1.0,
):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]  # 2
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]  # 3
        # probas = torch.softmax(logits, dim=-1)  # 4
        probas = softmax_with_temperature(logits, temperature)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # 5
        idx = torch.cat((idx, idx_next), dim=1)  # 6

    return idx


In [ ]:
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))